In [25]:
# set the matplotlib backend so figures can be saved in the background
import matplotlib
matplotlib.use("Agg")

In [26]:
# import the necessary packages
from keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import adam_v2
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.utils import to_categorical
from imutils import paths
import matplotlib.pyplot as plt
import numpy as np
import argparse
import random
import cv2
import os
import sys
import h5py
from keras.models import model_from_json  
sys.path.append('..')
#from lenet import LeNet

In [27]:
# import the necessary packages
from keras.models import load_model
import numpy as np
import random
import argparse
from imutils import paths
import cv2
import h5py
from keras.models import model_from_json 
from keras.callbacks import ModelCheckpoint
from skimage import exposure, color
from skimage.transform import resize

In [28]:
from keras.models import Sequential
from keras.layers.convolutional import Conv2D
from keras.layers.convolutional import MaxPooling2D
from keras.layers.core import Activation
from keras.layers.core import Flatten
from keras.layers.core import Dense
from keras.layers.core import Dropout
from keras.layers.normalization.batch_normalization_v1 import BatchNormalization
from keras import backend as K

In [29]:
from sklearn.model_selection import StratifiedKFold,KFold
from sklearn.model_selection import cross_val_score
from sklearn.utils import shuffle
from sklearn.metrics import r2_score

In [30]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [31]:
class LeNet:
    @staticmethod
    def build(width, height, depth):
        K.clear_session() 
        # initialize the model
        model = Sequential()
        inputShape = (height, width, depth)
        # if we are using "channels last", update the input shape
        if K.image_data_format() == "channels_first":   #for tensorflow
            inputShape = (depth, height, width)
        # first set of CONV => RELU => POOL layers
        model.add(Conv2D(32, (5, 5),padding="same",input_shape=inputShape))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()        
        #second set of CONV => RELU => POOL layers
        model.add(Conv2D(64, (5, 5), padding="same"))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()
        #third set of CONV => RELU => POOL layers
        model.add(Conv2D(128, (5, 5), padding="same"))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()
        #fourth set of CONV => RELU => POOL layers
        model.add(Conv2D(256, (5, 5), padding="same"))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()
        #fifth set of CONV => RELU => POOL layers() 128 512
        model.add(Conv2D(128, (5, 5), padding="same"))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        BatchNormalization()
        #sixth set of CONV => RELU => POOL layers
        #model.add(Conv2D(64, (5, 5), padding="same"))
        #model.add(Activation("relu"))
        #model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
        #BatchNormalization()
    
        # first (and only) set of FC => RELU layers
        model.add(Flatten())
        model.add(Dense(500))#500
        model.add(Activation("relu"))
        model.add(Dropout(0.5))#0.3 0.5
        
        model.add(Dense(128))
        model.add(Activation("relu"))
        model.add(Dropout(0.3))#0.25 0.3
        
        model.add(Dense(64))
        model.add(Activation("relu"))
        
        model.add(Dense(32))
        model.add(Activation("relu"))
        
        model.add(Dense(16))
        model.add(Activation("relu"))

        model.add(Dense(1))
        model.add(Activation("linear"))

        # return the constructed network architecture
        return model

In [32]:
# initialize the number of epochs to train for, initial learning rate,
# and batch size
seed=40
EPOCHS = 20
INIT_LR = 1e-3
BS = 8#48
#CLASS_NUM = 29
norm_size = 96

In [33]:
def load_data2(path):
    print("[INFO] loading images...")
    data = []
    labels = []
    # grab the image paths and randomly shuffle them
    imagePaths = sorted(list(paths.list_images(path)))
    random.seed(42)
    random.shuffle(imagePaths)
    # loop over the input images
    for imagePath in imagePaths:
        # load the image, pre-process it, and store it in the data list
        image = cv2.imread(imagePath)
        image = cv2.resize(image, (norm_size, norm_size))
        image = img_to_array(image)
        data.append(image)

        # extract the class label from the image path and update the
        # labels list
        label = float(imagePath.split(os.path.sep)[-2])       
        labels.append(label)  
        
    # scale the raw pixel intensities to the range [0, 1]
    X = np.array(data, dtype="float") / 255.0 
    #data = np.array(data, dtype="double")
    Y = np.array(labels)
    #print(Y)


    # partition the data into training and testing splits using 75% of
    # the data for training and the remaining 25% for testing
    #(X[train], X[test], Y[train], Y[test]) = train_test_split(data,labels, test_size=0.25, random_state=50)

    # convert the labels from integers to vectors
    #trainY = to_categorical(trainY, num_classes=CLASS_NUM)
    #testY = to_categorical(testY, num_classes=CLASS_NUM)   
    return X,Y

In [34]:
# In[*] shuffle in sklearn
def shuffle_skl(X,Y):
    X,Y = shuffle(X,Y, random_state=1337) 
#    print(X,Y)
    return X,Y

In [35]:
#def train(aug,trainX,trainY,testX,testY,args):
def train(aug,X,Y):
    
    for train, test in kfold.split(X, Y):
        # initialize the model
        print("[INFO] compiling model...")
        #reset_keras() 
        model = LeNet.build(width=norm_size, height=norm_size, depth=3)
        opt = adam_v2.Adam(learning_rate=INIT_LR, decay=INIT_LR / EPOCHS)
    
        #model.compile(optimizer='rmsprop',loss='mean_squared_error',metrics=['mae', 'acc'])
        model.compile(optimizer=opt,loss='mean_squared_error',metrics=['mae'])


        #reset_keras() 
        # train the network
        print("[INFO] loading network...")
        # load model (can train directly or train on the previous model)
        model = load_model('model_re_layer5_cycle11_7min_23651_scale70_resize96_nover_ta2.hdf5')
        print("[INFO] training network...")
        model_checkpoint = ModelCheckpoint('model_re_layer5_cycle11_7min_23651_scale70_resize96_nover_ta2.hdf5', monitor='val_mae',verbose=1, save_best_only=True)
        H = model.fit(aug.flow(X[train], Y[train], batch_size=BS),
        validation_data=(X[test], Y[test]), steps_per_epoch=len(X[train]) // BS,
        epochs=EPOCHS, verbose=1, shuffle=True, callbacks=[model_checkpoint])
    


In [36]:
if __name__=='__main__':
    #args = args_parse()
    X,Y = load_data2('Z:\\train_keras\\23651\\trainout\\cycle11_70_wnover2')
    #shuffle data
    X,Y = shuffle_skl(X,Y)
    # construct the image generator for data augmentation
    aug = ImageDataGenerator(rotation_range=30, width_shift_range=0,
        height_shift_range=0, shear_range=0.1, zoom_range=0,
        horizontal_flip=True, fill_mode="nearest")
    # define 10-fold cross validation test harness
    kfold = KFold(n_splits=10, shuffle=True, random_state=seed)
    cvscores = []
    
    #train(aug,trainX,trainY,testX,testY,args)
    train(aug,X,Y)
    print("[INFO] serializing network...")

[INFO] loading images...
[INFO] compiling model...
[INFO] loading network...
[INFO] training network...
Epoch 1/20
157/157 [==============================] - ETA: 0s - loss: 0.0360 - mae: 0.1231
Epoch 00001: val_mae improved from inf to 0.11718, saving model to model_re_layer5_cycle11_7min_23651_scale70_resize96_nover_ta2.hdf5
157/157 [==============================] - 10s 61ms/step - loss: 0.0360 - mae: 0.1231 - val_loss: 0.0262 - val_mae: 0.1172
Epoch 2/20
157/157 [==============================] - ETA: 0s - loss: 0.0363 - mae: 0.1240
Epoch 00002: val_mae improved from 0.11718 to 0.10992, saving model to model_re_layer5_cycle11_7min_23651_scale70_resize96_nover_ta2.hdf5
157/157 [==============================] - 7s 45ms/step - loss: 0.0363 - mae: 0.1240 - val_loss: 0.0265 - val_mae: 0.1099
Epoch 3/20
156/157 [============================>.] - ETA: 0s - loss: 0.0663 - mae: 0.1649
Epoch 00003: val_mae did not improve from 0.10992
157/157 [==============================] - 7s 43ms/step 

Epoch 10/20
156/157 [============================>.] - ETA: 0s - loss: 0.0284 - mae: 0.1112
Epoch 00010: val_mae did not improve from 0.09945
157/157 [==============================] - 7s 43ms/step - loss: 0.0285 - mae: 0.1116 - val_loss: 0.0300 - val_mae: 0.1212
Epoch 11/20
156/157 [============================>.] - ETA: 0s - loss: 0.0308 - mae: 0.1129
Epoch 00011: val_mae did not improve from 0.09945
157/157 [==============================] - 7s 44ms/step - loss: 0.0308 - mae: 0.1127 - val_loss: 0.0364 - val_mae: 0.1325
Epoch 12/20
157/157 [==============================] - ETA: 0s - loss: 0.0419 - mae: 0.1291
Epoch 00012: val_mae did not improve from 0.09945
157/157 [==============================] - 7s 43ms/step - loss: 0.0419 - mae: 0.1291 - val_loss: 0.0493 - val_mae: 0.1385
Epoch 13/20
156/157 [============================>.] - ETA: 0s - loss: 0.0355 - mae: 0.1223
Epoch 00013: val_mae did not improve from 0.09945
157/157 [==============================] - 7s 43ms/step - loss: 0.

Epoch 20/20
157/157 [==============================] - ETA: 0s - loss: 0.0281 - mae: 0.1112
Epoch 00020: val_mae did not improve from 0.08629
157/157 [==============================] - 26s 166ms/step - loss: 0.0281 - mae: 0.1112 - val_loss: 0.0189 - val_mae: 0.0898
[INFO] compiling model...
[INFO] loading network...
[INFO] training network...
Epoch 1/20
156/157 [============================>.] - ETA: 0s - loss: 0.0302 - mae: 0.1156
Epoch 00001: val_mae improved from inf to 0.08885, saving model to model_re_layer5_cycle11_7min_23651_scale70_resize96_nover_ta2.hdf5
157/157 [==============================] - 10s 60ms/step - loss: 0.0305 - mae: 0.1158 - val_loss: 0.0188 - val_mae: 0.0889
Epoch 2/20
157/157 [==============================] - ETA: 0s - loss: 0.0327 - mae: 0.1175
Epoch 00002: val_mae did not improve from 0.08885
157/157 [==============================] - 7s 42ms/step - loss: 0.0327 - mae: 0.1175 - val_loss: 0.0189 - val_mae: 0.0968
Epoch 3/20
156/157 [========================